In [ ]:
import shutil, os

# Copy file code dari input ke working directory
shutil.copy(
    '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/dataset_classification_vindr.py',
    '/kaggle/working/dataset_classification_vindr.py'
)

if os.path.exists('/kaggle/working/models'):
    shutil.rmtree('/kaggle/working/models')

shutil.copytree(
    '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/models',
    '/kaggle/working/models'
)

print("✅ File code berhasil di-copy ke /kaggle/working/")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms import v2 as transforms
from torch.utils.data import DataLoader
import time
import numpy as np
from sklearn import metrics
from sklearn.preprocessing import label_binarize

In [ ]:
## Import dataset modules and model
from dataset_classification_vindr import MakeDataset_VinDr_classification
from models.mvswintransformer import MVSwinTransformer

In [ ]:
## Devices
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Available GPUs: ", torch.cuda.device_count())
print("Current device ID: ", torch.cuda.current_device())

In [ ]:
## Configuration
extension   = ".png"
target_size = 224   # gambar sudah 224x224 dari preprocessing
window_size = 7     # window 7x7 untuk input 224x224

In [ ]:
# Define hyperparameters
batch_size    = 16
learning_rate = 1e-4
epochs        = 50
weight_decay  = 1e-3
alpha         = 1.0   # bobot loss BI-RADS
beta          = 1.0   # bobot loss Density

In [ ]:
## Data Paths
import os

# Deteksi otomatis: Kaggle atau lokal
if os.path.exists("/kaggle/input"):
    # Path di Kaggle (ada 2 level dataset_preprocessed)
    image_dir     = "/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/dataset_preprocessed/dataset_preprocessed"
    label_dir_csv = "/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/breast-level_annotations.csv"
else:
    # Path lokal
    image_dir     = "./dataset_preprocessed"
    label_dir_csv = "./breast-level_annotations.csv"

print("Image dir    :", image_dir)
print("Label CSV    :", label_dir_csv)
print("Image dir exists:", os.path.exists(image_dir))
print("CSV exists   :", os.path.exists(label_dir_csv))

In [ ]:
## Data Loaders

# Transform TRAIN: augmentasi lengkap (gambar sudah 224x224 dari preprocessing)
transform_train = transforms.Compose([
    transforms.ToTensor(),                                      # (H,W,C) → (C,H,W), scale [0,1]
    transforms.RandomHorizontalFlip(p=0.5),                    # Random Horizontal Flip
    transforms.RandomVerticalFlip(p=0.5),                      # Random Vertical Flip
    transforms.RandomRotation(degrees=15),                     # Random Rotation ±15°
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),  # Gaussian Blur
    transforms.ColorJitter(brightness=0.2),                    # Brightness Adjustment ±20%
])

# Transform TEST: tanpa augmentasi
transform_test = transforms.Compose([
    transforms.ToTensor(),
])

# Dataset langsung dari kolom split di CSV (training=80%, test=20%)
train_dataloader = MakeDataset_VinDr_classification(
    image_dir     = image_dir,
    label_dir_csv = label_dir_csv,
    transform     = transform_train,
    mode          = 'train',
    target_size   = target_size
)

test_dataloader = MakeDataset_VinDr_classification(
    image_dir     = image_dir,
    label_dir_csv = label_dir_csv,
    transform     = transform_test,
    mode          = 'test',
    target_size   = target_size
)

train_loader = DataLoader(train_dataloader, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataloader,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataloader)} pasangan | Test: {len(test_dataloader)} pasangan")

In [ ]:
# Create model instance
model = MVSwinTransformer(img_size=224, window_size=7).to(device)

In [ ]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of parameters: ", pytorch_total_params // 10 ** 6, " mil")
print("Total number of trainable parameters: ", pytorch_total_trainable_params // 10 ** 6, " mil")

In [ ]:
## Hitung class weights dari training set (Weighted CCE untuk imbalance)
def compute_class_weights(dataset, num_classes, label_key):
    counts = torch.zeros(num_classes)
    for pair in dataset.pairs:
        counts[pair[label_key]] += 1
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * num_classes  # normalize
    return weights

weights_birads  = compute_class_weights(train_dataloader, 5, 'label_birads').to(device)
weights_density = compute_class_weights(train_dataloader, 4, 'label_density').to(device)

print("Class weights BI-RADS :", weights_birads)
print("Class weights Density :", weights_density)

## Optimizer, Scheduler, Loss
optimizer         = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler         = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)  # ReduceLROnPlateau: butuh .step(metric) — pakai train_loss
criterion_birads  = nn.CrossEntropyLoss(weight=weights_birads)
criterion_density = nn.CrossEntropyLoss(weight=weights_density)

print(f"Scheduler type: {type(scheduler).__name__}")

In [ ]:
# Path simpan model
model_save_path = "/kaggle/working/best_model.pth" if os.path.exists("/kaggle/input") else "./best_model.pth"
print("Model akan disimpan di:", model_save_path)


best_train_loss = float('inf')

# Training loop
for epoch in range(1, epochs + 1):
    since = time.time()
    print('-' * 50)
    print(f"Epoch [{epoch}/{epochs}]")

    # ── TRAIN ──────────────────────────────────────────
    model.train()
    running_loss    = 0.0
    correct_birads  = 0
    correct_density = 0
    total           = 0

    for inputs_cc, inputs_mlo, labels_birads, labels_density in train_loader:
        inputs_cc      = inputs_cc.float().to(device)
        inputs_mlo     = inputs_mlo.float().to(device)
        labels_birads  = labels_birads.long().to(device)
        labels_density = labels_density.long().to(device)

        optimizer.zero_grad()

        pred_birads, pred_density = model(inputs_cc, inputs_mlo)

        loss_birads  = criterion_birads(pred_birads,   labels_birads)
        loss_density = criterion_density(pred_density, labels_density)
        total_loss   = alpha * loss_birads + beta * loss_density

        total_loss.backward()
        optimizer.step()

        running_loss    += total_loss.item()
        total           += labels_birads.size(0)
        correct_birads  += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
        correct_density += (pred_density.argmax(dim=1) == labels_density).sum().item()

    train_loss        = running_loss / len(train_loader)
    train_acc_birads  = 100 * correct_birads  / total
    train_acc_density = 100 * correct_density / total
    time_elapsed      = time.time() - since
    curr_lr           = optimizer.param_groups[0]["lr"]

    print(f"  [TRAIN] Loss: {train_loss:.4f} | "
          f"Acc BI-RADS: {train_acc_birads:.2f}% | "
          f"Acc Density: {train_acc_density:.2f}% | "
          f"LR: {curr_lr:.6f} | "
          f"Time: {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s")

    # ReduceLROnPlateau: wajib pakai .step(metric), gunakan train_loss
    scheduler.step(train_loss)

    # Simpan model terbaik berdasarkan train loss
    if train_loss < best_train_loss:
        best_train_loss = train_loss
        torch.save(model.state_dict(), model_save_path)
        print(f"  ✅ Model terbaik disimpan (train_loss: {train_loss:.4f})")


In [ ]:
# ── EVALUASI TEST SET ──────────────────────────────────
print("=" * 50)
print("Evaluasi Test Set")
print("=" * 50)

model.load_state_dict(torch.load(model_save_path))
model.eval()

correct_birads   = 0
correct_density  = 0
total            = 0
all_prob_birads  = []
all_prob_density = []
all_true_birads  = []
all_true_density = []

with torch.no_grad():
    for inputs_cc, inputs_mlo, labels_birads, labels_density in test_loader:
        inputs_cc      = inputs_cc.float().to(device)
        inputs_mlo     = inputs_mlo.float().to(device)
        labels_birads  = labels_birads.long().to(device)
        labels_density = labels_density.long().to(device)

        pred_birads, pred_density = model(inputs_cc, inputs_mlo)

        total            += labels_birads.size(0)
        correct_birads   += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
        correct_density  += (pred_density.argmax(dim=1) == labels_density).sum().item()

        all_prob_birads.append(F.softmax(pred_birads,   dim=1).cpu().numpy())
        all_prob_density.append(F.softmax(pred_density, dim=1).cpu().numpy())
        all_true_birads.append(labels_birads.cpu().numpy())
        all_true_density.append(labels_density.cpu().numpy())

test_acc_birads  = 100 * correct_birads  / total
test_acc_density = 100 * correct_density / total

all_prob_birads  = np.concatenate(all_prob_birads,  axis=0)
all_prob_density = np.concatenate(all_prob_density, axis=0)
all_true_birads  = np.concatenate(all_true_birads,  axis=0)
all_true_density = np.concatenate(all_true_density, axis=0)

auc_birads  = metrics.roc_auc_score(
    label_binarize(all_true_birads,  classes=[0,1,2,3,4]),
    all_prob_birads,  multi_class='ovr', average='macro'
)
auc_density = metrics.roc_auc_score(
    label_binarize(all_true_density, classes=[0,1,2,3]),
    all_prob_density, multi_class='ovr', average='macro'
)

print(f"  BI-RADS → Accuracy: {test_acc_birads:.2f}%  | AUC (macro-OvR): {auc_birads:.4f}")
print(f"  Density → Accuracy: {test_acc_density:.2f}% | AUC (macro-OvR): {auc_density:.4f}")